# 📘 Module 6.2 — VLMs for ADAS Applications

**Goal:** Explore how Vision-Language Models are used in ADAS systems.

## VLM Applications in ADAS

```
Camera Feed → VLM → Scene Understanding
                  → Risk Assessment
                  → Natural Language Explanations
                  → Anomaly Detection
```

---

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 1. Architecture: LLaVA-style VLM

LLaVA (Large Language and Vision Assistant) connects a vision encoder to an LLM:

```
Image → ViT Encoder → Visual Tokens → Projection → ┐
                                                      ├→ LLM → Response
Text  → Tokenizer   → Text Tokens  ──────────────→ ┘
```

In [2]:
class SimpleVLM(nn.Module):
    """Simplified VLM architecture (LLaVA-style)."""
    
    def __init__(self, vision_dim=768, text_dim=512, vocab_size=10000):
        super().__init__()
        
        # Vision encoder (simplified ViT)
        self.vision_encoder = nn.Sequential(
            nn.Conv2d(3, 256, 16, stride=16),  # Patch embedding
            nn.Flatten(2),
        )
        
        # Vision-to-language projection (bridges the two modalities)
        self.vision_proj = nn.Sequential(
            nn.Linear(256, text_dim),
            nn.GELU(),
            nn.Linear(text_dim, text_dim),
        )
        
        # Text embedding
        self.text_embed = nn.Embedding(vocab_size, text_dim)
        
        # Language model (simplified)
        self.lm = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=text_dim, nhead=8, dim_feedforward=2048,
                batch_first=True
            ),
            num_layers=4
        )
        
        self.output_head = nn.Linear(text_dim, vocab_size)
    
    def forward(self, image, text_tokens):
        # Encode image to visual tokens
        visual_features = self.vision_encoder(image)  # (B, 256, N_patches)
        visual_features = visual_features.transpose(1, 2)  # (B, N_patches, 256)
        visual_tokens = self.vision_proj(visual_features)  # (B, N_patches, text_dim)
        
        # Encode text
        text_embeddings = self.text_embed(text_tokens)  # (B, seq_len, text_dim)
        
        # Concatenate visual + text tokens
        combined = torch.cat([visual_tokens, text_embeddings], dim=1)
        
        # Process through language model
        output = self.lm(combined)
        
        # Get text output (skip visual tokens)
        text_output = output[:, visual_tokens.size(1):, :]
        logits = self.output_head(text_output)
        
        return logits

# Test
vlm = SimpleVLM()
image = torch.randn(2, 3, 224, 224)
text = torch.randint(0, 10000, (2, 20))  # 20 text tokens

output = vlm(image, text)
print(f"Image: {image.shape}")
print(f"Text tokens: {text.shape}")
print(f"Output logits: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in vlm.parameters()):,}")

Image: torch.Size([2, 3, 224, 224])
Text tokens: torch.Size([2, 20])
Output logits: torch.Size([2, 20, 10000])
Parameters: 23,450,640


## 2. ADAS Use Cases for VLMs

In [3]:
# --- ADAS VLM Use Cases ---

use_cases = {
    "Scene Description": {
        "prompt": "Describe this driving scene in detail.",
        "response": "A four-lane urban road with moderate traffic. Two vehicles are "
                    "in the adjacent lane. A pedestrian is waiting at the crosswalk "
                    "on the right. The traffic light ahead is green.",
        "adas_use": "Driver awareness, logging, incident reconstruction"
    },
    "Risk Assessment": {
        "prompt": "What are the potential hazards in this scene?",
        "response": "1. Pedestrian may step into the road unexpectedly. "
                    "2. Vehicle in left lane is drifting. "
                    "3. Construction sign ahead suggests lane narrowing.",
        "adas_use": "Proactive safety warnings, driver alerts"
    },
    "Anomaly Detection": {
        "prompt": "Is anything unusual in this driving scene?",
        "response": "Yes — there is an object on the road ahead that appears "
                    "to be debris. Also, the vehicle ahead has its hazard lights on, "
                    "suggesting it may be stopping.",
        "adas_use": "Edge case detection, unknown object handling"
    },
}

for case_name, case in use_cases.items():
    print(f"📋 {case_name}")
    print(f"   Prompt:   {case['prompt']}")
    print(f"   Response: {case['response']}")
    print(f"   ADAS Use: {case['adas_use']}")
    print()

📋 Scene Description
   Prompt:   Describe this driving scene in detail.
   Response: A four-lane urban road with moderate traffic. Two vehicles are in the adjacent lane. A pedestrian is waiting at the crosswalk on the right. The traffic light ahead is green.
   ADAS Use: Driver awareness, logging, incident reconstruction

📋 Risk Assessment
   Prompt:   What are the potential hazards in this scene?
   Response: 1. Pedestrian may step into the road unexpectedly. 2. Vehicle in left lane is drifting. 3. Construction sign ahead suggests lane narrowing.
   ADAS Use: Proactive safety warnings, driver alerts

📋 Anomaly Detection
   Prompt:   Is anything unusual in this driving scene?
   Response: Yes — there is an object on the road ahead that appears to be debris. Also, the vehicle ahead has its hazard lights on, suggesting it may be stopping.
   ADAS Use: Edge case detection, unknown object handling



## 3. Open-Vocabulary Detection

Traditional detectors can only find classes they were trained on. VLM-based detectors can find **anything** described in text.

In [4]:
# --- Open Vocabulary vs. Closed Vocabulary ---
print("Traditional Object Detector:")
print("  Trained classes: [car, truck, pedestrian, cyclist]")
print("  ✅ Can detect: car, truck, pedestrian, cyclist")
print("  ❌ Cannot detect: shopping cart, fallen tree, animal")
print()
print("VLM-Based Open-Vocabulary Detector:")
print("  Text query: 'shopping cart on the road'")
print("  ✅ Can detect it! (even though never trained on shopping carts)")
print()
print("This is CRITICAL for ADAS:")
print("  → The real world has infinite object categories")
print("  → Edge cases involve unusual objects")
print("  → VLMs can detect novel/rare objects using text descriptions")

Traditional Object Detector:
  Trained classes: [car, truck, pedestrian, cyclist]
  ✅ Can detect: car, truck, pedestrian, cyclist
  ❌ Cannot detect: shopping cart, fallen tree, animal

VLM-Based Open-Vocabulary Detector:
  Text query: 'shopping cart on the road'
  ✅ Can detect it! (even though never trained on shopping carts)

This is CRITICAL for ADAS:
  → The real world has infinite object categories
  → Edge cases involve unusual objects
  → VLMs can detect novel/rare objects using text descriptions


---
## ✅ Key Takeaways

1. **VLMs for ADAS** enable scene understanding, risk assessment, and anomaly detection
2. **LLaVA architecture** connects a vision encoder to an LLM via projection
3. **Open-vocabulary detection** handles novel objects not in training data
4. VLMs can provide **natural language explanations** of driving decisions
5. Challenges: latency, hallucination, and deployment on edge devices

---
## 📖 Next Steps
➡️ **Next module:** [07_VLAs/01_vla_concepts.ipynb](../07_VLAs/01_vla_concepts.ipynb) — Vision-Language-Action models for driving